# Package

**TimeGym's code lives in these notebooks, so a contributor needs to know which cell to edit and how that edit safely becomes package code.**

This chapter writes two small files the rest of the package leans on: `timegym/_internal.py`, a few guards shared by many modules, and `timegym/__init__.py`, which decides what `import timegym` gives you.

Below we see what the shared guards refuse and find the notebook that owns each public name. Then, in a throwaway project, we export one of this chapter's cells and watch the check catch a generated file that no longer matches its notebook.

**Development notebook.** Expand the collapsed cells to edit the code, then restart the kernel and Run All ([build workflow](README.md#develop-the-package-here)).

In [1]:
#| default_exp __init__

In [2]:
from contextlib import redirect_stderr, redirect_stdout
import importlib.util
import io
import json
from pathlib import Path
import sys
from tempfile import TemporaryDirectory

import nbformat
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'pyproject.toml').is_file() and (p / 'timegym').is_dir())
pd.set_option('display.max_colwidth', None)

## Shared guards

Many modules need the same small checks, so they live here once and a fix applies everywhere. These cells write `timegym/_internal.py`, which is private: sibling modules import from it, users shouldn't rely on it.

- `positive_int` and `nonnegative_int` accept whole numbers only, so a seed or a horizon is never a `bool` or a float in disguise.
- `_real_array` turns data into a float array but refuses complex numbers instead of dropping their imaginary part.
- `_json_safe` turns evidence into plain JSON: NumPy numbers into Python numbers, arrays into lists, dates into text, an exception into its type, message and traceback. It refuses NaN and infinity.

In [3]:
#| export _internal
from dataclasses import fields, is_dataclass
from enum import Enum
import math
import traceback
from typing import Any
import numpy as np


def _integer(value: Any, name: str, *, minimum: int) -> int:
    if isinstance(value, (bool, np.bool_)) or not isinstance(value, (int, np.integer)) or value < minimum:
        qualifier = "positive" if minimum else "nonnegative"
        raise ValueError(f"{name} must be a {qualifier} integer")
    return int(value)


def positive_int(value: Any, name: str) -> int:
    return _integer(value, name, minimum=1)


def nonnegative_int(value: Any, name: str) -> int:
    return _integer(value, name, minimum=0)

In [4]:
#| export _internal
def _has_complex_values(array: np.ndarray) -> bool:
    """Detect complex scalars even when an object array hides their dtype."""
    return np.iscomplexobj(array) or (
        array.dtype.kind == "O" and any(np.iscomplexobj(value) for value in array.flat)
    )

In [5]:
#| export _internal
def _real_array(value: Any, name: str) -> np.ndarray:
    """Convert real numeric data without discarding an imaginary component."""
    array = np.asarray(value)
    if _has_complex_values(array):
        raise ValueError(f"{name} must contain real-valued numbers")
    return np.asarray(array, dtype=float)

In [6]:
#| export _internal
def _exception_evidence(exc: BaseException) -> dict:
    """Format the supplied exception even outside its handling block."""
    return {
        "exception_type": type(exc).__name__,
        "exception_message": str(exc),
        "traceback": "".join(traceback.format_exception(exc)),
    }


def _json_safe(value):
    """Canonicalize evidence to JSON values, rejecting nonfinite numbers."""
    if isinstance(value, Enum):
        return _json_safe(value.value)
    if value is None:
        return None
    if isinstance(value, str):
        return str(value)
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (int, np.integer)):
        return int(value)
    if isinstance(value, (float, np.floating)):
        value = float(value)
        if not math.isfinite(value):
            raise ValueError("JSON evidence contains a nonfinite number")
        return float(value)
    if isinstance(value, np.generic):
        return _json_safe(value.item())
    if isinstance(value, np.ndarray):
        return _json_safe(value.tolist())
    if isinstance(value, BaseException):
        return _exception_evidence(value)
    if is_dataclass(value):
        return {item.name: _json_safe(getattr(value, item.name)) for item in fields(value)}
    if isinstance(value, dict):
        return {str(key): _json_safe(item) for key, item in value.items()}
    if isinstance(value, (tuple, list)):
        return [_json_safe(item) for item in value]
    if hasattr(value, "isoformat"):
        return str(value.isoformat())
    return repr(value)


### What the guards refuse

Each row is one call and what came back:

In [7]:
def outcome(function, *args):
    try:
        return repr(function(*args))
    except ValueError as refusal:
        return f'refused: {refusal}'


calls = [
    (positive_int, 3, 'horizon'),
    (positive_int, np.int64(3), 'horizon'),
    (positive_int, True, 'horizon'),
    (positive_int, 3.0, 'horizon'),
    (nonnegative_int, -1, 'seed'),
    (_real_array, [1.5, 2], 'point'),
    (_real_array, np.array([1 + 2j]), 'point'),
    (_json_safe, {'mse': np.float64(0.25), 'steps': np.arange(3), 'when': pd.Timestamp('2026-01-01')}),
    (_json_safe, RuntimeError('model crashed')),
    (_json_safe, {'mse': float('nan')}),
]
guards = pd.DataFrame([{'call': f"{function.__name__}({', '.join(map(repr, args))})",
                        'result': outcome(function, *args)} for function, *args in calls])
assert guards['result'][2].startswith('refused')  # True is not the integer 1 here
assert guards['result'][6].startswith('refused')  # a complex number is refused, not truncated
guards

,call,result
0,"positive_int(3, 'horizon')",3
1,"positive_int(np.int64(3), 'horizon')",3
2,"positive_int(True, 'horizon')",refused: horizon must be a positive integer
3,"positive_int(3.0, 'horizon')",refused: horizon must be a positive integer
4,"nonnegative_int(-1, 'seed')",refused: seed must be a nonnegative integer
5,"_real_array([1.5, 2], 'point')","array([1.5, 2. ])"
6,"_real_array(array([1.+2.j]), 'point')",refused: point must contain real-valued numbers
7,"_json_safe({'mse': np.float64(0.25), 'steps': array([0, 1, 2]), 'when': Timestamp('2026-01-01 00:00:00')})","{'mse': 0.25, 'steps': [0, 1, 2], 'when': '2026-01-01T00:00:00'}"
8,_json_safe(RuntimeError('model crashed')),"{'exception_type': 'RuntimeError', 'exception_message': 'model crashed', 'traceback': 'RuntimeError: model crashed\n'}"
9,_json_safe({'mse': nan}),refused: JSON evidence contains a nonfinite number


In plain Python `True` counts as the integer 1, and NumPy turns a complex array into floats by dropping the imaginary part with only a warning. Both would let a bug through quietly; here they stop with a message that names the argument. The last three rows are how evidence enters a report: as plain values a JSON file can hold, with a crash kept as its type, message and traceback.

## What `import timegym` gives you

The public entry point is one cell. It imports the names a user needs from the modules that define them, sets the version and lists the names in `__all__`. Add a name here only when users should reach it as `timegym.Name`; everything else stays public at its module path. Leave `__version__` alone: the release workflow sets it when a release is published.

In [8]:
#| export __init__
from timegym.contracts import Forecast, ForecastTask, Reference, SimulationReference
from timegym.scenarios import Scenario, Case
from timegym.pipeline import Pipeline
from timegym.models import NaiveForecaster, LastValueForecaster, GaussianAR1Forecaster
from timegym.reporting import TestReport, DiagnosticStatus, DiagnosticCategory, DiagnosticResult, CaseResult
from timegym.evaluation import ForecastRun, EvaluatedCase
from timegym.grid import PATTERNS, GridCell, scenario_grid
from timegym.scorecard import Scorecard

__version__ = "1.3.0"
__all__ = ["Case", "Forecast", "ForecastTask", "Reference", "SimulationReference", "Scenario", "Pipeline",
           "NaiveForecaster", "LastValueForecaster", "GaussianAR1Forecaster", "TestReport", "ForecastRun", "EvaluatedCase", "DiagnosticStatus", "DiagnosticCategory", "DiagnosticResult", "CaseResult",
           "Scorecard", "PATTERNS", "GridCell", "scenario_grid"]

### Which notebook do I edit?

Each public name comes from one module, and the [source map](source-map.json) gives each module one owning notebook. Put the two together and you know where to make a change:

In [9]:
# Inspect the entry point defined above, not a previously imported copy.
assert all(name in globals() for name in __all__)  # every public name is defined
print(f'{len(__all__)} public names')

owners = {module: entry['notebook'] for entry in json.loads((ROOT / 'source' / 'source-map.json').read_text())['sources']
          for module in entry['modules']}


def defined_in(name):
    """The module a public name comes from. A plain value such as PATTERNS has no __module__."""
    value = globals()[name]
    return getattr(value, '__module__', None) or next(
        module for module, loaded in sys.modules.items()
        if module.startswith('timegym.') and name in getattr(loaded, '__all__', ()))


homes = pd.DataFrame([{'module': defined_in(name), 'public name': name} for name in __all__])
homes['edit in'] = homes['module'].map(lambda module: Path(owners[module.replace('.', '/') + '.py']).name)
homes.groupby(['edit in', 'module'])['public name'].agg(', '.join).to_frame()

21 public names


public name
edit in                     module                                                                                            
01_synthetic_data.ipynb     timegym.scenarios                                                                         Scenario
06_forecasting_models.ipynb timegym.models                         NaiveForecaster, LastValueForecaster, GaussianAR1Forecaster
07_pipeline.ipynb           timegym.evaluation                                                      ForecastRun, EvaluatedCase
                            timegym.pipeline                                                                          Pipeline
08_forecast_contracts.ipynb timegym.contracts                     Case, Forecast, ForecastTask, Reference, SimulationReference
10_reports_and_replay.ipynb timegym.reporting   TestReport, DiagnosticStatus, DiagnosticCategory, DiagnosticResult, CaseResult
13_scorecard.ipynb          timegym.grid                                                     PATTERNS, GridCell, scenario_grid
                            timegym.scorecard                                                                        Scorecard

So to change `Scorecard`, edit notebook 13. To change `Case`, edit notebook 08: the entry point imports it through `timegym.scenarios`, but it is defined in `timegym.contracts`. Everything else is public at its module path, such as the data-preparation helpers in `timegym.features`, `timegym.target`, `timegym.smoothing` and `timegym.data.real`; the [API reference](../docs/reference.md) lists them with their optional dependencies.

## From a notebook edit to package code

`python tools/export_notebooks.py` reads the **saved** notebooks and writes one module for each export destination. It never runs a cell, and running a notebook never writes a module. To watch the whole loop without touching the real package, we copy two cells of this chapter into a throwaway project: the integer guards from above, and the setup cell, which is a demonstration and not exported. Then we run the repository's exporter on it.

In [10]:
# Load tools/export_notebooks.py as a module: each run below calls the same main() as the command line.
spec = importlib.util.spec_from_file_location('export_notebooks', ROOT / 'tools' / 'export_notebooks.py')
exporter = sys.modules[spec.name] = importlib.util.module_from_spec(spec)
spec.loader.exec_module(exporter)


def run_exporter(project, *options):
    """Like `python tools/export_notebooks.py --root project ...`: return the exit status and message."""
    printed = io.StringIO()
    with redirect_stdout(printed), redirect_stderr(printed):
        status = exporter.main(['--root', str(project), *options])
    return status, printed.getvalue().strip()

In [11]:
chapter = nbformat.read(ROOT / 'source' / '12_package.ipynb', as_version=4)  # the saved file, as export reads it
by_id = {cell.id: cell for cell in chapter.cells}

scratch = TemporaryDirectory(prefix='timegym-build-')
project = Path(scratch.name)
(project / 'source').mkdir()
(project / 'timegym').mkdir()
notebook_path = project / 'source' / '12_package.ipynb'
module_path = project / 'timegym' / '_internal.py'
copy = nbformat.v4.new_notebook(metadata=chapter.metadata,
                                cells=[by_id['12_package-02'], by_id['shared-integer-validation']])
nbformat.write(copy, notebook_path)
(project / 'source' / 'source-map.json').write_text(json.dumps({
    'schema_version': 1, 'handwritten': [],
    'sources': [{'notebook': 'source/12_package.ipynb', 'modules': ['timegym/_internal.py']}]}))

status, message = run_exporter(project)
print(message, end='\n\n')
generated = module_path.read_text()
print(generated)
print('The real timegym/_internal.py starts with exactly these lines:',
      (ROOT / 'timegym' / '_internal.py').read_text().startswith(generated))

Built 1 notebook-owned modules; 1 changed.

# AUTOGENERATED! DO NOT EDIT! File to edit: ../source/12_package.ipynb.

from dataclasses import fields, is_dataclass
from enum import Enum
import math
import traceback
from typing import Any
import numpy as np

__all__ = ["positive_int", "nonnegative_int"]

# %% ../source/12_package.ipynb #shared-integer-validation


def _integer(value: Any, name: str, *, minimum: int) -> int:
    if (
        isinstance(value, (bool, np.bool_))
        or not isinstance(value, (int, np.integer))
        or value < minimum
    ):
        qualifier = "positive" if minimum else "nonnegative"
        raise ValueError(f"{name} must be a {qualifier} integer")
    return int(value)


def positive_int(value: Any, name: str) -> int:
    return _integer(value, name, minimum=1)


def nonnegative_int(value: Any, name: str) -> int:
    return _integer(value, name, minimum=0)

The real timegym/_internal.py starts with exactly these lines: True


The first line says where to edit. The imports move to the top, `__all__` lists the public functions, Black formats the code, and a `# %%` line names the notebook and cell id each block came from, so any line of `timegym/*.py` leads back to its cell. The setup cell is not there. Because export is deterministic, these lines are the start of the real `timegym/_internal.py`.

### The check catches drift

`--check` builds the modules the saved notebooks would produce and compares them, byte for byte, with the files on disk. We try the same small change in the wrong place and in the right place: first by hand in the generated file, then in the notebook.

In [12]:
steps = []


def record(what, *options):
    status, message = run_exporter(project, *options)
    steps.append({'what we did': what, 'exit status': status, 'what the exporter says': ' '.join(message.split())})


record('Nothing since the export; check', '--check')
module_path.write_text(module_path.read_text().replace('integer"', 'whole number"'))
record('Edited the generated file by hand; check', '--check')
record('Exported again')
assert 'whole number' not in module_path.read_text()  # export overwrote the hand edit
copy.cells[1].source = copy.cells[1].source.replace('integer"', 'whole number"')
nbformat.write(copy, notebook_path)
record('Edited the notebook and saved it, without exporting; check', '--check')
record('Exported')
scratch.cleanup()
assert [step['exit status'] for step in steps] == [0, 1, 0, 1, 0]  # drift fails the check either way
pd.DataFrame(steps).set_index('what we did')

,exit status,what the exporter says
what we did,,
Nothing since the export; check,0,Checked 1 notebook-owned modules; 0 changed.
Edited the generated file by hand; check,1,"Generated modules differ from their source notebooks: timegym/_internal.py Save the notebooks, then run python tools/export_notebooks.py."
Exported again,0,Built 1 notebook-owned modules; 1 changed.
"Edited the notebook and saved it, without exporting; check",1,"Generated modules differ from their source notebooks: timegym/_internal.py Save the notebooks, then run python tools/export_notebooks.py."
Exported,0,Built 1 notebook-owned modules; 1 changed.


A hand edit to a generated file fails the check, and the next export erases it, so it is never the place for a fix. A saved notebook edit fails the check too, until you export it. Exit status 1 means drift; status 2 means the notebooks themselves can't be built, for example a cell without an id or a notebook without its `source` role.

## In this repository

The same commands, from the repository root:

```bash
python tools/export_notebooks.py          # write timegym/*.py from the saved notebooks
python tools/export_notebooks.py --check  # "0 changed": the package matches the notebooks
python -m pytest -q                       # test the generated package
python tools/check_notebooks.py source/12_package.ipynb   # run one chapter in a fresh kernel
```

Save the notebook before exporting. After editing a cell, restart the kernel and Run All, so the examples use the new definitions. If you change code that another chapter owns, save and export that notebook, then restart this kernel before importing it.

**Limits.** The check proves that the package matches the notebooks, not that the code is right: the tests and a fresh-kernel run of every notebook do that. `_internal` is private, so its helpers may change without notice; sibling modules import them from `timegym._internal` directly.

## Change one thing

- Add a name to `__all__` without importing it: the first assert under "Which notebook do I edit?" fails.
- Copy a second cell into the throwaway project, such as `_json_safe` (`10_reports_and_repla-09`), and export: the module gains a second block headed by that cell's id.
- Remove the `timegym` role from the throwaway notebook's metadata and export: the exporter refuses with exit status 2 and says why.

**Next:** [13 · Scorecard](13_scorecard.ipynb). The [build guide](../tools/README.md) covers the source map and same-chapter bindings in full.